**Step 1: Environment Setup & Library Imports**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Create the main project folder and a data folder inside it
project_path = '/content/drive/MyDrive/GenAI_Assignment_1'
data_path = f'{project_path}/data'

os.makedirs(data_path, exist_ok=True)
print(f"Created project folder at: {project_path}")
print(f"Created data folder at: {data_path}")

Created project folder at: /content/drive/MyDrive/GenAI_Assignment_1
Created data folder at: /content/drive/MyDrive/GenAI_Assignment_1/data


In [ ]:
!pip install -q langchain langchain-community langchain-groq sentence-transformers chromadb faiss-cpu pypdf
print("Installation Complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.

In [ ]:
!pip install -q langchain-text-splitters

**Step 2: Document Ingestion**

**Step 3: Text Chunking Strategy**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Make sure this matches the name of the file you uploaded to Drive!
pdf_file_path = '/content/drive/MyDrive/GenAI_Assignment_1/data/syllabus.pdf'

# 1. Load the Document
print(f"Loading document from: {pdf_file_path}")
loader = PyPDFLoader(pdf_file_path)
documents = loader.load()
print(f"Total pages loaded: {len(documents)}")

# 2. Chunking Strategy A: Small Chunks
text_splitter_small = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
small_chunks = text_splitter_small.split_documents(documents)
print(f"Generated {len(small_chunks)} small chunks.")

# 3. Chunking Strategy B: Large Chunks
text_splitter_large = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)
large_chunks = text_splitter_large.split_documents(documents)
print(f"Generated {len(large_chunks)} large chunks.")

Loading document from: /content/drive/MyDrive/GenAI_Assignment_1/data/syllabus.pdf
Total pages loaded: 6
Generated 31 small chunks.
Generated 14 large chunks.


**Step 4: Embedding Models Initialization**

**Step 5: Vector Database Construction**

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma, FAISS

# 1. Setup Embedding Model A (Small dimension: 384)
print("Downloading MiniLM embeddings (384 dimensions)...")
embedding_model_small = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Setup Embedding Model B (Larger dimension: 768)
print("Downloading BGE embeddings (768 dimensions)...")
embedding_model_large = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

# 3. Vector Database 1: ChromaDB (Using small chunks and small embeddings)
print("Building ChromaDB Vector Store...")
persist_directory = '/content/drive/MyDrive/GenAI_Assignment_1/chroma_db'
chroma_db = Chroma.from_documents(
    documents=small_chunks,
    embedding=embedding_model_small,
    persist_directory=persist_directory
)
print("ChromaDB successfully created and saved to Drive!")

# 4. Vector Database 2: FAISS (Using large chunks and large embeddings)
print("Building FAISS Vector Store...")
faiss_db = FAISS.from_documents(
    documents=large_chunks,
    embedding=embedding_model_large
)
# Save FAISS locally to drive
faiss_db.save_local('/content/drive/MyDrive/GenAI_Assignment_1/faiss_index')
print("FAISS database successfully created and saved to Drive!")

/tmp/ipykernel_3407/3161006133.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model_small = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building ChromaDB Vector Store...
ChromaDB successfully created and saved to Drive!
Building FAISS Vector Store...
FAISS database successfully created and saved to Drive!


In [ ]:
!pip install -q langchain-classic

**Step 6: LLM Setup & Prompt Engineering**

**Step 7: RAG Pipeline Configuration**

In [ ]:
import os
import getpass
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Securely Input Groq API Key
print("Please paste your Groq API Key below (it will be hidden):")
os.environ["GROQ_API_KEY"] = getpass.getpass()

# 2. Initialize the different LLMs from your Groq
print("\nConnecting to Groq LLMs...")
llm_llama = ChatGroq(model_name="llama-3.1-8b-instant")
# Using the GPT-style model. (If Groq gives an access error, change this to "gemma2-9b-it" or "mixtral-8x7b-32768")
llm_gpt = ChatGroq(model_name="openai/gpt-oss-120b")

# 3. Prompt Engineering
# This defines the persona and injects the retrieved chunks
system_prompt = (
    "You are a helpful, professional University Course Assistant. "
    "Use the following pieces of retrieved context to answer the student's question accurately. "
    "If the answer is not contained within the context, simply state 'I do not have information about that in the current course documents.' Do not guess.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. Create the Complete RAG Chains (Combining DB + LLM)
# Configuration 1: ChromaDB (Small Chunks) + LLaMA-3.1
question_answer_chain_1 = create_stuff_documents_chain(llm_llama, prompt)
rag_config_1 = create_retrieval_chain(
    chroma_db.as_retriever(search_kwargs={"k": 3}), # Retrieve top 3 chunks
    question_answer_chain_1
)

# Configuration 2: FAISS (Large Chunks) + GPT-OSS model
question_answer_chain_2 = create_stuff_documents_chain(llm_gpt, prompt)
rag_config_2 = create_retrieval_chain(
    faiss_db.as_retriever(search_kwargs={"k": 2}), # Retrieve top 2 large chunks
    question_answer_chain_2
)

print("✅ Step 6 Complete! Both RAG Configurations are ready.")

Please paste your Groq API Key below (it will be hidden):
··········

Connecting to Groq LLMs...
✅ Step 6 Complete! Both RAG Configurations are ready.


**Step 8: System Testing & Example Outputs**

In [ ]:
# Step 7: Testing the RAG System & Comparing Outputs

def ask_course_assistant(question):
    print(f"\n🎓 User Question: {question}")
    print("=" * 70)

    # ---------------------------------------------------------
    # Configuration 1: ChromaDB (Small Chunks) + LLaMA-3.1
    # ---------------------------------------------------------
    print("🤖 Configuration 1: LLaMA-3.1-8B | Small Chunks | ChromaDB")
    print("-" * 70)
    try:
        response_1 = rag_config_1.invoke({"input": question})
        print(f"Answer: {response_1['answer']}\n")
    except Exception as e:
         print(f"Error with Model 1: {e}\n")

    # ---------------------------------------------------------
    # Configuration 2: FAISS (Large Chunks) + GPT-OSS 120B
    # ---------------------------------------------------------
    print("-" * 70)
    print("🤖 Configuration 2: GPT-OSS-120B | Large Chunks | FAISS")
    print("-" * 70)
    try:
        response_2 = rag_config_2.invoke({"input": question})
        print(f"Answer: {response_2['answer']}\n")
    except Exception as e:
         print(f"Error with Model 2: {e}\n")
    print("=" * 70)

# Questions derived directly from your uploaded syllabus
test_questions = [
    "What is the total weightage for Assignment 1 & 2, and how will they be evaluated?",
    "What specific topics are covered in Unit 7: Retrieval-Augmented & Tool-Enhanced LLM Workflows?",
    "What is the recourse examination policy if a student fails this course?"
]

# Run the test loop
print("Starting Evaluation of RAG Configurations...\n")
for q in test_questions:
    ask_course_assistant(q)

Starting Evaluation of RAG Configurations...


🎓 User Question: What is the total weightage for Assignment 1 & 2, and how will they be evaluated?
🤖 Configuration 1: LLaMA-3.1-8B | Small Chunks | ChromaDB
----------------------------------------------------------------------
Answer: I do not have information about that in the current course documents.

----------------------------------------------------------------------
🤖 Configuration 2: GPT-OSS-120B | Large Chunks | FAISS
----------------------------------------------------------------------
Answer: The two programming assignments together account for **40 % of the total course grade** (each assignment is worth 20 %).  

**Evaluation method:**  
- The assignments are administered as continuous‑assessment tasks.  
- They are evaluated offline.  
- After submitting the code, each student must **demonstrate** the solution and **explain** the approach in a **demo** session, which is followed by a **viva (oral questioning)**.  

Both ass